Scenario: Customer Support Call Center

A company runs a support chatbot that needs to route customer queries to the right department. Instead of one big script, they design a state graph where each node represents a specialized agent.

1. State Definition (SupportState)

The chatbot keeps track of:
- query → What the customer asked.
- category → Which department it belongs to (billing, tech, general).
- response → What the bot replies with.

Think of this as the customer’s “ticket form.”

2. Router Node (route_query)
 When a customer types a question, the router scans it:
 - If the query mentions “bill” or “payment”, it routes to billing_agent.
 - If it mentions “error” or “bug”, it routes to tech_agent.
 - Otherwise, it defaults to general_agent.
 This is like a receptionist deciding which desk you should go to.

 3. Agent Nodes
 - billing_agent → Replies with “Billing dept: [query]”.
 - tech_agent → Replies with “Tech support: [query]”.
 - general_agent → Replies with “General help: [query]”.
 Each agent specializes in its own type of problem.

 4. Graph Flow
 - Entry point: router.
 - Router decides the path based on the query.
 - The query flows into the correct agent node.
 - The agent generates a response and ends the conversation.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class SupportState(TypedDict):
    query: str
    category: str   # "billing" | "tech" | "general"
    response: str

# Router: reads state, returns next node name
def route_query(state: SupportState) -> str:
    q = state["query"].lower()
    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

def billing_agent(state):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state):
    return {"response": "General help: " + state["query"]}


# Build graph with conditional routing
g = StateGraph(SupportState)
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)


# One entry point routes to 3 different nodes!
g.set_entry_point("router")
g.add_conditional_edges(
    "router",    # from node
    route_query, # function that returns next node
    {            # mapping: return value → node name
        "billing_agent":  "billing_agent",
        "tech_agent":     "tech_agent",
        "general_agent":  "general_agent",
    }
)
